In [0]:
from pyspark.sql.types import StructType, StructField,StringType, IntegerType, DateType, TimestampType, FloatType 

import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
# Define schema table for the data file 

brand_schema = StructType([ 

    StructField('brand_code',StringType(),False), #require field 

    StructField ('brand_name', StringType(). True), 

    StructField ('category_code', StringType(), True) 

]) 

###Create Spark Data frame in memorry

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw_data/brands/*.csv" 


df= spark.read.option("header", "true").option("delimeter",",").schema(brand_schema).csv(raw_data_path) 
#add metada columns 
df = df.withColumn("_source_file",F.col("_metadata.file_path")) \ 
    .withColumn("ingested_at",F.current_timestamp()) 
display(df.limit(5)) 

In [0]:
df.write.format("delta").mode("overwrite") \ 
    .option("mergeSchema", "true")\ 
    .saveAsTable(f"{catalog_name}.bronze.brz_brands") 

#Create Category Delta Table


In [0]:
category_schema=StructType([
    StructField("category_code",StringType(),False), # false means that the field is required
    StructField("category_name", StringType(), True)
])

###Create Data Spark Frame in Memorry


In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw_data/category/*.csv"

df = spark.read.option("header",True).option \
("delimeter",",").schema(category_schema).csv(raw_data_path)# true means to specify the first row as a header

#Add metadata columns
df=df.withColumn("ingested_at",F.current_timestamp())\
    .withColumn("_source_file",F.col("_metadata.file_path"))

display(df.limit(5))

###Create Delta Table 

In [0]:
df.write.format("delta").mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_category")

#Create Products


In [0]:
products_schema = StructType([
    StructField ("product_id",StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(),True),
    StructField("length_cm", StringType(), True),
    StructField("width_cm", FloatType(), True),
    StructField("height_cm",FloatType(), True ),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(),False),
    StructField("ingest_timestamp", TimestampType(), False)
])


###Create Spark in memorry

In [0]:
# load data using the source file path
raw_data_path ="/Volumes/ecommerce/source_data/raw_data/products/*.csv";

df= spark.read.option("header",True).option("delimiter",",")\
    .schema(products_schema)\
        .csv(raw_data_path)\
            .withColumn("file_name",F.col("_metadata.file_path"))\
                .withColumn("ingest_timestamp", F.current_timestamp())


In [0]:
df.limit(5).display()

###Create Delta Table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeschema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_products")